# APCS 例題練習

本 Notebook 蒐集數個與課程主題相呼應的練習題，著重在 APCS 常見題型：運算思維、字串處理、模擬與綜合分析。
建議先自行思考再參考解法，必要時可複製題目到新的 Code Cell 嘗試。

---
**範例結構**
- 題目敘述與輸入輸出說明
- 思考提示
- 參考解答與測試案例
- 延伸討論建議


## 例題 1：小組成績統計

**題目**
- 已知一個列表 `students`，每位學生含 `name`、`quiz_scores`、`lab_scores`。
- 小考與實作權重分別為 70%、30%。請輸出每位學生的加權成績與班級平均。
- 另需將學生依加權成績由高至低排序。

**輸出範例**
```
姓名    加權成績
Aaron    86.90
……
班級平均 84.52
```

**思考提示**
- 可將 `sum(...) / len(...)` 包成函式，避免重複計算。
- 利用 `sorted` 的 `key` 參數進行排序。


In [1]:
students = [
    {"name": "Aaron", "quiz_scores": [78, 83, 90], "lab_scores": [92, 88, 95]},
    {"name": "Kevin", "quiz_scores": [91, 85, 88], "lab_scores": [97, 93, 90]},
    {"name": "Chloe", "quiz_scores": [68, 74, 70], "lab_scores": [80, 84, 79]},
    {"name": "Shawn", "quiz_scores": [88, 92, 90], "lab_scores": [85, 90, 92]},
    {"name": "Ethan", "quiz_scores": [75, 78, 82], "lab_scores": [88, 85, 90]},
]

def weighted_score(student, w_quiz=0.7, w_lab=0.3):
    """計算單一學生的小考/實作加權成績。"""
    quiz_avg = sum(student["quiz_scores"]) / len(student["quiz_scores"])
    lab_avg = sum(student["lab_scores"]) / len(student["lab_scores"])
    return w_quiz * quiz_avg + w_lab * lab_avg

results = []
for stu in students:
    score = weighted_score(stu)
    results.append({"name": stu["name"], "score": score})

sorted_results = sorted(results, key=lambda item: item["score"], reverse=True)
class_average = sum(item["score"] for item in sorted_results) / len(sorted_results)

print("姓名    加權成績")
for item in sorted_results:
    print(f"{item['name']:<4}  {item['score']:.2f}")
print(f"班級平均 {class_average:.2f}")

assert len(sorted_results) == len(students)
assert sorted_results[0]["score"] >= sorted_results[-1]["score"]


姓名    加權成績
Shawn    89.70
Kevin    89.60
Aaron    86.07
Ethan    81.13
Chloe    73.77
班級平均 84.05


**延伸討論**
- 自行調整權重或增加評量面向，如專題成績。
- 將結果輸出成 `list` 或 `dict`，供後續統計/視覺化使用。
- 評估若提供補考機制，如何更新加權公式。


## 例題 2：字串壓縮 (Run-Length Encoding)

**題目**
- 給定一個只包含大寫字母的字串，如 `AABBBCCDAA`。
- 將連續且相同的字元壓縮為「字元 + 連續次數」，輸出 `A2B3C2D1A2`。
- 若輸入為空字串，回傳空字串。

**思考提示**
- 逐一檢查字元，紀錄目前字元與連續次數。
- 當字元改變時，把前一段結果加入清單。


In [2]:
def run_length_encode(text: str) -> str:
    """以 run-length encoding 壓縮連續字元。"""
    if not text:
        return ""

    segments = []
    current_char = text[0]
    count = 1

    for ch in text[1:]:
        if ch == current_char:
            count += 1
        else:
            segments.append(f"{current_char}{count}")
            current_char = ch
            count = 1

    segments.append(f"{current_char}{count}")
    return "".join(segments)

sample = "AABBBCCDAA"
encoded = run_length_encode(sample)
print(sample, "→", encoded)

assert encoded == "A2B3C2D1A2"
assert run_length_encode("ABCD") == "A1B1C1D1"
assert run_length_encode("") == ""


AABBBCCDAA → A2B3C2D1A2


**延伸討論**
- 計算壓縮後長度是否真的比原字串短。
- 若要支援大小寫混合，須調整資料結構或函式簽章。
- 思考如何將壓縮結果還原成原字串。


## 例題 3：單伺服器排隊模擬

**題目**
- 每筆任務含 `id`、`arrival`（抵達時間）、`duration`（處理時間）。
- 伺服器一次只能處理一個任務，採先到先服務（FCFS）。
- 求每個任務的開始時間、結束時間與等待時間，並輸出平均等待時間。

**思考提示**
- 使用一個變數追蹤伺服器目前的可用時間。
- `start_time = max(arrival, server_available_time)`。
- 將結果儲存於列表後格式化輸出。


In [3]:
tasks = [
    {"id": "T1", "arrival": 0, "duration": 3},
    {"id": "T2", "arrival": 2, "duration": 4},
    {"id": "T3", "arrival": 5, "duration": 2},
    {"id": "T4", "arrival": 6, "duration": 1},
]

timeline = []
server_available_time = 0

for task in tasks:
    start_time = max(task["arrival"], server_available_time)
    finish_time = start_time + task["duration"]
    wait_time = start_time - task["arrival"]
    timeline.append({
        "id": task["id"],
        "start": start_time,
        "finish": finish_time,
        "wait": wait_time,
    })
    server_available_time = finish_time

print("任務  開始  結束  等待")
for entry in timeline:
    print(f"{entry['id']:<3}  {entry['start']:>3}   {entry['finish']:>3}   {entry['wait']:>3}")

average_wait = sum(item["wait"] for item in timeline) / len(timeline)
print(f"平均等待時間：{average_wait:.1f}")

assert timeline[0]["wait"] == 0
assert timeline[-1]["finish"] == server_available_time


任務  開始  結束  等待
T1     0     3     0
T2     3     7     1
T3     7     9     2
T4     9    10     3
平均等待時間：1.5


**延伸討論**
- 嘗試改成雙伺服器，觀察平均等待時間變化。
- 若任務允許排隊超過一定秒數就放棄，需如何調整資料結構？
- 與 APCS 模擬題對照：確保輸入讀取、排序策略與邊界條件皆已處理。


## 例題 4：區間合併與總長度

**題目**
- 輸入多段 `[start, end)` 時間區間，代表學生投入練習的時段。
- 合併所有重疊或相接的區間後，輸出整理後的區間列表與總練習時長。
- 例：`[(0, 30), (15, 45), (60, 90), (80, 120)]` → 合併後 `[(0, 45), (60, 120)]`，總長度 105。

**思考提示**
- 先依起點排序。
- 逐一比較下一段是否與當前區間重疊或相接。


In [4]:
from typing import List, Tuple

Interval = Tuple[int, int]

def merge_intervals(intervals: List[Interval]) -> List[Interval]:
    """合併所有重疊或相接的閉左開右區間。"""
    if not intervals:
        return []

    sorted_intervals = sorted(intervals, key=lambda x: x[0])
    merged: List[Interval] = [sorted_intervals[0]]

    for start, end in sorted_intervals[1:]:
        last_start, last_end = merged[-1]
        if start <= last_end:  # 重疊或相接
            merged[-1] = (last_start, max(last_end, end))
        else:
            merged.append((start, end))

    return merged

def total_length(intervals: List[Interval]) -> int:
    return sum(end - start for start, end in intervals)

study_sessions = [(0, 30), (15, 45), (60, 90), (80, 120)]
merged_sessions = merge_intervals(study_sessions)
duration = total_length(merged_sessions)

print("原始區間:", study_sessions)
print("合併後  :", merged_sessions)
print("總練習時長:", duration)

assert merged_sessions == [(0, 45), (60, 120)]
assert duration == 105
assert merge_intervals([]) == []


原始區間: [(0, 30), (15, 45), (60, 90), (80, 120)]
合併後  : [(0, 45), (60, 120)]
總練習時長: 105


**延伸討論**
- APCS 常見題型會在最後加上查詢：例如給定一段時間，判斷是否有人在練習。
- 將區間改成二维平面（如排程 + 地點），需要什麼資料結構？
- 試著將輸入改為由檔案或標準輸入讀取。


---
若你完成以上練習，可挑選 APCS 歷屆題逐題對照，或將自己的解答整理進這個 Notebook 以累積題庫。
歡迎在 `notebooks/` 目錄新增更多主題頁。
